# Layer 4: K-Means and DBSCAN Reviewer Clustering

This notebook groups Yelp reviewers into behavioural clusters using two complementary
unsupervised algorithms — K-Means (partitional) and DBSCAN (density-based) — applied
to engineered reviewer-level features.  The goal is to surface tightly-knit clusters
whose members share high-spam behavioural profiles, as well as extreme outlier accounts
that fall outside any natural grouping, all **without** using the spam label during
clustering so the analysis remains purely unsupervised and free of label leakage.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Imports ready.")

## 1. Feature Preparation

Several raw behavioural features (`total_reviews`, `mean_review_length`,
`mean_exclamations`) are heavily right-skewed, so we apply a **log1p** transform
before scaling to compress the long tail and bring the distributions closer to
Gaussian — a condition that benefits K-Means' Euclidean distance metric.

After the log transform every feature is centred and unit-scaled with
`StandardScaler` so that no single dimension dominates the distance calculations.

**Important:** `spam_rate` is deliberately excluded from the clustering feature
matrix.  Including it would leak the very label we later use for evaluation,
producing artificially clean clusters that cannot generalise.  We retain it only
as a post-hoc evaluation column.

In [ ]:
df = pd.read_csv("reviewer_features.csv")
print(f"Loaded {df.shape[0]:,} reviewers, {df.shape[1]} columns.")

CLUSTER_FEATURES = [
    "total_reviews", "mean_rating", "rating_std",
    "mean_review_length", "mean_exclamations", "mean_caps_ratio",
    "concentration",
]

LOG_FEATURES = ["total_reviews", "mean_review_length", "mean_exclamations"]

feat = df[CLUSTER_FEATURES].copy()
for col in LOG_FEATURES:
    feat[col] = np.log1p(feat[col])

scaler = StandardScaler()
X = scaler.fit_transform(feat)
X = pd.DataFrame(X, columns=CLUSTER_FEATURES, index=df.index)

print(f"Scaled feature matrix shape: {X.shape}")
X.sample(5, random_state=42)

## 2. K-Means — Choosing K

We sweep K from 2 to 12 and record two diagnostics at each step:

* **Inertia (SSE)** — the within-cluster sum of squares; an *elbow* in this
  curve signals diminishing returns from additional clusters.
* **Silhouette score** — measures how similar each point is to its own cluster
  versus the nearest neighbouring cluster (range −1 to 1; higher is better).

The recommended K is selected as the value that maximises the silhouette score.

In [ ]:
K_RANGE = range(2, 13)
inertias, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

best_k = list(K_RANGE)[np.argmax(sil_scores)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_RANGE), inertias, "o-", color="steelblue")
axes[0].axvline(best_k, ls="--", color="tomato", label=f"K = {best_k}")
axes[0].set(xlabel="K", ylabel="Inertia (SSE)", title="Elbow Curve")
axes[0].legend()

axes[1].plot(list(K_RANGE), sil_scores, "s-", color="seagreen")
axes[1].axvline(best_k, ls="--", color="tomato", label=f"K = {best_k}")
axes[1].set(xlabel="K", ylabel="Silhouette Score", title="Silhouette Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nRecommended K = {best_k}  (silhouette = {max(sil_scores):.4f})")

## 3. K-Means Final Clustering

We fit K-Means with the recommended K and attach cluster labels back to the
original dataframe.  The summary table below reports the size, mean spam rate,
and average behavioural features per cluster so we can profile each group.

In [ ]:
km_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
df["kmeans_cluster"] = km_final.fit_predict(X)

summary_cols = ["spam_rate"] + CLUSTER_FEATURES
km_summary = (
    df.groupby("kmeans_cluster")[summary_cols]
    .agg(["mean", "count"])
    .droplevel(1, axis=1)
)

km_summary = df.groupby("kmeans_cluster").agg(
    size=("user_id", "count"),
    spam_rate_mean=("spam_rate", "mean"),
    total_reviews_mean=("total_reviews", "mean"),
    mean_rating_mean=("mean_rating", "mean"),
    rating_std_mean=("rating_std", "mean"),
    mean_review_length_mean=("mean_review_length", "mean"),
    mean_exclamations_mean=("mean_exclamations", "mean"),
    mean_caps_ratio_mean=("mean_caps_ratio", "mean"),
    concentration_mean=("concentration", "mean"),
)

display(km_summary.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
km_summary["spam_rate_mean"].plot.bar(ax=ax, color="coral", edgecolor="black")
ax.set(xlabel="K-Means Cluster", ylabel="Mean Spam Rate",
       title="Spam Rate by K-Means Cluster")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 4. DBSCAN — Density-Based Clustering

DBSCAN does not require a pre-specified K.  Instead it discovers clusters as
dense regions separated by sparser zones, and labels points that do not belong
to any dense region as **noise** (label −1).  This makes it especially useful
for isolating extreme outlier reviewers whose feature vectors sit far from any
natural grouping.

We choose `eps` via a k-distance plot and set `min_samples = 2 × n_features`
as a robust starting heuristic.

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 2 * X.shape[1]

nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(X)
distances, _ = nn.kneighbors(X)
k_distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(k_distances, color="steelblue")
ax.set(xlabel="Points (sorted)", ylabel=f"{min_samples}-NN Distance",
       title="k-Distance Plot for eps Selection")
plt.tight_layout()
plt.show()

knee_idx = int(len(k_distances) * 0.95)
eps_value = float(np.round(k_distances[knee_idx], 2))
print(f"Chosen eps = {eps_value}  (95th-percentile k-distance)")
print(f"min_samples = {min_samples}")

dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
df["dbscan_cluster"] = dbscan.fit_predict(X)

n_clusters_db = df["dbscan_cluster"].nunique() - (1 if -1 in df["dbscan_cluster"].values else 0)
n_noise = (df["dbscan_cluster"] == -1).sum()
print(f"\nDBSCAN found {n_clusters_db} cluster(s) + {n_noise:,} noise points.")

db_summary = df.groupby("dbscan_cluster").agg(
    size=("user_id", "count"),
    spam_rate_mean=("spam_rate", "mean"),
    total_reviews_mean=("total_reviews", "mean"),
    mean_rating_mean=("mean_rating", "mean"),
    concentration_mean=("concentration", "mean"),
)
display(db_summary.round(3))

df["is_noise"] = (df["dbscan_cluster"] == -1).astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="is_noise", y="spam_rate", ax=ax,
            palette={0: "lightblue", 1: "salmon"})
ax.set_xticklabels(["Cluster Members", "Noise / Outliers"])
ax.set(xlabel="", ylabel="Spam Rate",
       title="Spam Rate: DBSCAN Cluster Members vs Noise Points")
plt.tight_layout()
plt.show()

## 5. Cluster Behavioral Profiles

The heatmap below visualises the **mean scaled feature value** for each K-Means
cluster.  Warm colours indicate features that are elevated relative to the
population mean; cool colours indicate suppression.  This makes it easy to read
off each cluster's dominant behavioural signature at a glance.

In [ ]:
X_with_cluster = X.copy()
X_with_cluster["kmeans_cluster"] = df["kmeans_cluster"].values

cluster_profiles = X_with_cluster.groupby("kmeans_cluster")[CLUSTER_FEATURES].mean()

fig, ax = plt.subplots(figsize=(10, max(3, best_k * 0.7)))
sns.heatmap(
    cluster_profiles,
    annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    linewidths=0.5, ax=ax,
)
ax.set(title="K-Means Cluster Profiles (mean scaled features)",
       xlabel="Feature", ylabel="Cluster")
plt.tight_layout()
plt.show()

## 6. Key Findings

* **Cluster [X] (size [N], spam rate [Y]%):** Characterised by low review
  count, low rating standard deviation, and high seller concentration —
  consistent with single-target throwaway accounts used in coordinated
  spammer campaigns.

* **Cluster [X] (size [N], spam rate [Y]%):** High review volume, moderate
  ratings, low concentration — resembles genuine power reviewers with the
  lowest spam incidence.

* **Cluster [X] (size [N], spam rate [Y]%):** Extreme ratings (very low or
  very high mean), elevated caps ratio and exclamation use — emotional or
  promotional writing style linked to above-average spam.

* **Cluster [X] (size [N], spam rate [Y]%):** Short review length, moderate
  concentration — borderline group whose spam rate sits between the cleanest
  and dirtiest clusters.

* **DBSCAN noise points ([N] reviewers, spam rate [Y]%):** Extreme outliers
  isolated from every density cluster — these carry the highest spam
  concentration in the dataset, confirming that points unreachable by any
  cluster are disproportionately fraudulent.

## 7. Handoff to L5

Three cluster-derived features are exported for the Layer 5 ensemble classifier:

| Feature | Type | Description |
|---|---|---|
| `kmeans_cluster` | Categorical (one-hot or label-encoded) | Partitions the reviewer space into K behavioural groups; the classifier can learn cluster-specific spam propensities. |
| `dbscan_noise_flag` | Binary (0 / 1) | Flags reviewers that DBSCAN classified as noise — a strong standalone spam signal given the elevated spam rate observed among noise points. |
| `cluster_spam_rate` | Continuous [0, 1] | The mean spam rate of the reviewer's K-Means cluster, acting as a **soft prior** — a smoothed, population-level estimate of spam propensity for the reviewer's behavioural neighbourhood. |

Together these features let the downstream model leverage both the hard
partitioning of K-Means and the outlier-detection power of DBSCAN without
requiring a separate clustering step at inference time (cluster assignment
is deterministic given the trained centroids and eps/min_samples).